In [ ]:
{
  "cells": [
    {
      "cell_type": "markdown",
      "id": "c00",
      "metadata": {
        "id": "c00"
      },
      "source": [
        "# Proyecto Integrador: Scouting con Machine Learning\n",
        "## ¿Cuánto vale un futbolista? ¿Puedes adivinar su posición?\n",
        "### Curso: Inteligencia Artificial y Aprendizaje Automático\n",
        "\n",
        "---\n",
        "\n",
        "| Campo | Información |\n",
        "|-------|-------------|\n",
        "| **Nombre** | _________________________ |\n",
        "| **Código** | _________________________ |\n",
        "| **Fecha de entrega** | _________________________ |\n",
        "\n",
        "---\n",
        "\n",
        "### Contexto\n",
        "\n",
        "EA Sports asigna a cada jugador de fútbol real una serie de atributos numéricos en FIFA 23.\n",
        "Estos datos (de **SoFIFA**) reflejan las habilidades reales de más de 16 000 futbolistas.\n",
        "\n",
        "Como analista de datos de un club, debes responder dos preguntas:\n",
        "\n",
        "\u003E **Tarea 1 – Regresión:** ¿En cuánto debería cotizarse un jugador en el mercado de fichajes?\n",
        "\n",
        "\u003E **Tarea 2 – Clasificación:** ¿Cuál es la posición de un jugador (GK/DEF/MID/FWD) según sus stats?\n",
        "\n",
        "---\n",
        "\n",
        "**Descarga del dataset**\n",
        "\n",
        "1. Ve a Kaggle → busca *\"FIFA 23 Complete Player Dataset\"* del usuario `stefanoleone992`\n",
        "2. Descarga `players_23.csv` y colócalo en la misma carpeta que este notebook\n",
        "3. Alternativa CLI: `kaggle datasets download -d stefanoleone992/fifa-23-complete-player-dataset`\n"
      ]
    },
    {
      "cell_type": "markdown",
      "id": "c01",
      "metadata": {
        "id": "c01"
      },
      "source": [
        "## Objetivos y estructura\n",
        "\n",
        "| Parte | Tema (Notebook de referencia) | Qué harás |\n",
        "|-------|------------------------------|-----------|\n",
        "| **1** | Carga y exploración (NB-01) | Cargar el dataset, explorar las variables |\n",
        "| **2** | Preparación de datos (NB-01) | Limpiar, crear `position_cat`, hacer el split |\n",
        "| **3** | Regresión (NB-02) | Predecir **`value_eur`** (valor de mercado) |\n",
        "| **4** | Clasificación (NB-03) | Predecir **`position_cat`** (GK/DEF/MID/FWD) |\n",
        "| **5** | Ensamble (NB-04) | Mejorar con Random Forest |\n",
        "| **6** | Validación y ajuste (NB-05) | Afinar el mejor modelo con GridSearchCV |\n",
        "\n",
        "### Features que usaremos\n",
        "\n",
        "| Variable | Descripción | Rol |\n",
        "|----------|-------------|-----|\n",
        "| `age`, `height_cm`, `weight_kg` | Físico | Feature |\n",
        "| `overall`, `potential` | Calificación general / potencial | Feature |\n",
        "| `pace`, `shooting`, `passing` | Stats de ataque | Feature |\n",
        "| `dribbling`, `defending`, `physic` | Stats de juego | Feature |\n",
        "| `weak_foot`, `skill_moves`, `international_reputation` | Habilidades | Feature |\n",
        "| `value_eur` | Valor de mercado en € | **Target regresión** |\n",
        "| `position_cat` (creada de `player_positions`) | GK / DEF / MID / FWD | **Target clasificación** |\n",
        "\n",
        "### Los 6 ejercicios\n",
        "\n",
        "| # | Ejercicio | Tema | Dificultad |\n",
        "|---|-----------|------|------------|\n",
        "| 1 | EDA del valor de mercado + preparar features | Preparación | ⭐⭐☆ |\n",
        "| 2 | Árbol de Decisión para regresión | Regresión | ⭐⭐☆ |\n",
        "| 3 | KNN para regresión | Regresión | ⭐⭐☆ |\n",
        "| 4 | Árbol de Decisión + KNN para clasificación | Clasificación | ⭐⭐⭐ |\n",
        "| 5 | Random Forest para clasificación + importancia | Ensamble | ⭐⭐⭐ |\n",
        "| 6 | GridSearchCV para afinar el mejor modelo | Validación | ⭐⭐⭐ |\n"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "id": "c02",
      "metadata": {
        "id": "c02"
      },
      "outputs": [],
      "source": [
        "# ============================================================\n",
        "# PARTE 0: IMPORTACIÓN DE LIBRERÍAS\n",
        "# ============================================================\n",
        "\n",
        "import numpy as np\n",
        "import pandas as pd\n",
        "import matplotlib.pyplot as plt\n",
        "import seaborn as sns\n",
        "import warnings\n",
        "warnings.filterwarnings('ignore')\n",
        "\n",
        "# Preprocesamiento\n",
        "from sklearn.model_selection import (train_test_split, cross_val_score,\n",
        "                                     GridSearchCV, StratifiedKFold)\n",
        "from sklearn.preprocessing import StandardScaler\n",
        "from sklearn.pipeline import Pipeline\n",
        "\n",
        "# Regresión\n",
        "from sklearn.linear_model import LinearRegression\n",
        "from sklearn.tree import DecisionTreeRegressor\n",
        "from sklearn.neighbors import KNeighborsRegressor\n",
        "from sklearn.ensemble import RandomForestRegressor\n",
        "\n",
        "# Clasificación\n",
        "from sklearn.linear_model import LogisticRegression\n",
        "from sklearn.tree import DecisionTreeClassifier\n",
        "from sklearn.neighbors import KNeighborsClassifier\n",
        "from sklearn.ensemble import RandomForestClassifier\n",
        "\n",
        "# Métricas\n",
        "from sklearn.metrics import (mean_squared_error, mean_absolute_error, r2_score,\n",
        "                              accuracy_score, classification_report, confusion_matrix)\n",
        "\n",
        "# Configuración visual\n",
        "plt.rcParams['figure.figsize'] = (10, 6)\n",
        "plt.rcParams['figure.dpi'] = 100\n",
        "sns.set_style('whitegrid')\n",
        "pd.set_option('display.max_columns', 20)\n",
        "\n",
        "RANDOM_STATE = 42\n",
        "np.random.seed(RANDOM_STATE)\n",
        "\n",
        "print('Librerías importadas correctamente.')\n"
      ]
    },
    {
      "cell_type": "markdown",
      "id": "c03",
      "metadata": {
        "id": "c03"
      },
      "source": [
        "---\n",
        "# Parte 1: Carga y Exploración Inicial\n",
        "\n",
        "\u003E **Conexión con Notebook 01** — Antes de construir cualquier modelo debemos entender los datos:\n",
        "\u003E cuántos hay, qué variables tienen, si hay valores faltantes, y cómo se distribuyen.\n"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "id": "c04",
      "metadata": {
        "id": "c04"
      },
      "outputs": [],
      "source": [
        "# === CARGA DEL DATASET ===\n",
        "# Si usas Google Colab sube el archivo manualmente:\n",
        "#   from google.colab import files; files.upload()\n",
        "\n",
        "try:\n",
        "    df_raw = pd.read_csv('players_23.csv', low_memory=False)\n",
        "    print(f'Dataset cargado: {df_raw.shape[0]:,} jugadores x {df_raw.shape[1]} columnas')\n",
        "except FileNotFoundError:\n",
        "    print('ERROR: Archivo players_23.csv no encontrado.')\n",
        "    print('Descargalo de: kaggle datasets download -d stefanoleone992/fifa-23-complete-player-dataset')\n"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "id": "c05",
      "metadata": {
        "id": "c05"
      },
      "outputs": [],
      "source": [
        "# === SELECCIÓN DE COLUMNAS RELEVANTES ===\n",
        "\n",
        "COLUMNAS = [\n",
        "    'short_name', 'age', 'nationality_name', 'club_name',\n",
        "    'player_positions', 'overall', 'potential',\n",
        "    'value_eur', 'wage_eur', 'height_cm', 'weight_kg',\n",
        "    'pace', 'shooting', 'passing', 'dribbling', 'defending', 'physic',\n",
        "    'weak_foot', 'skill_moves', 'international_reputation'\n",
        "]\n",
        "\n",
        "COLUMNAS_OK = [c for c in COLUMNAS if c in df_raw.columns]\n",
        "df = df_raw[COLUMNAS_OK].copy()\n",
        "\n",
        "print(f'Columnas seleccionadas: {len(COLUMNAS_OK)}')\n",
        "print(f'Dimensiones finales:    {df.shape[0]:,} x {df.shape[1]}')\n",
        "print()\n",
        "df.head()\n"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "id": "c06",
      "metadata": {
        "id": "c06"
      },
      "outputs": [],
      "source": [
        "# === EDA INICIAL ===\n",
        "\n",
        "print('TOP 10 JUGADORES (por Overall)')\n",
        "print('=' * 70)\n",
        "top10 = df[['short_name', 'overall', 'potential', 'value_eur',\n",
        "            'player_positions', 'club_name']].nlargest(10, 'overall')\n",
        "print(top10.to_string(index=False))\n",
        "\n",
        "fig, axes = plt.subplots(1, 2, figsize=(14, 5))\n",
        "\n",
        "axes[0].hist(df['overall'].dropna(), bins=30, color='cornflowerblue',\n",
        "             edgecolor='navy', alpha=0.8)\n",
        "axes[0].axvline(df['overall'].mean(), color='red', linestyle='--', linewidth=2,\n",
        "                label=f\"Media: {df['overall'].mean():.1f}\")\n",
        "axes[0].set_xlabel('Overall (calificación general)', fontsize=12)\n",
        "axes[0].set_ylabel('Jugadores', fontsize=12)\n",
        "axes[0].set_title('Distribución del Overall', fontsize=13)\n",
        "axes[0].legend()\n",
        "\n",
        "axes[1].hist(df['age'].dropna(), bins=25, color='darkorange',\n",
        "             edgecolor='brown', alpha=0.8)\n",
        "axes[1].axvline(df['age'].mean(), color='red', linestyle='--', linewidth=2,\n",
        "                label=f\"Media: {df['age'].mean():.1f}\")\n",
        "axes[1].set_xlabel('Edad', fontsize=12)\n",
        "axes[1].set_ylabel('Jugadores', fontsize=12)\n",
        "axes[1].set_title('Distribución de Edades', fontsize=13)\n",
        "axes[1].legend()\n",
        "\n",
        "plt.tight_layout()\n",
        "plt.show()\n"
      ]
    },
    {
      "cell_type": "markdown",
      "id": "c07",
      "metadata": {
        "id": "c07"
      },
      "source": [
        "---\n",
        "# Parte 2: Preparación de Datos\n",
        "\n",
        "\u003E **Conexión con Notebook 01** — Limpiamos el dataset, manejamos valores faltantes y\n",
        "\u003E creamos la variable `position_cat` (target de clasificación).\n",
        "\n",
        "Las dos celdas siguientes ya están implementadas. **Léelas con atención.**\n"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "id": "c08",
      "metadata": {
        "id": "c08"
      },
      "outputs": [],
      "source": [
        "# === [CÓDIGO COMPLETO] LIMPIEZA DE DATOS ===\n",
        "\n",
        "print('Valores faltantes antes de limpiar:')\n",
        "print(df.isnull().sum()[df.isnull().sum() \u003E 0])\n",
        "\n",
        "# 1. Solo jugadores con valor de mercado conocido (\u003E 0)\n",
        "df = df[df['value_eur'] \u003E 0].copy()\n",
        "\n",
        "# 2. Solo jugadores con posición conocida\n",
        "df = df[df['player_positions'].notna()].copy()\n",
        "\n",
        "# 3. Eliminar filas con NaN en las 6 stats principales\n",
        "STATS_PRINCIPALES = ['pace', 'shooting', 'passing', 'dribbling', 'defending', 'physic']\n",
        "df = df.dropna(subset=STATS_PRINCIPALES).copy()\n",
        "\n",
        "# 4. Rellenar NaN restantes con la mediana\n",
        "cols_num = df.select_dtypes(include=[np.number]).columns\n",
        "df[cols_num] = df[cols_num].fillna(df[cols_num].median())\n",
        "\n",
        "print(f'\\nDataset limpio: {df.shape[0]:,} jugadores')\n",
        "print(f'Valores faltantes restantes: {df.isnull().sum().sum()}')\n"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "id": "c09",
      "metadata": {
        "id": "c09"
      },
      "outputs": [],
      "source": [
        "# === [CÓDIGO COMPLETO] FEATURE ENGINEERING: CATEGORÍA DE POSICIÓN ===\n",
        "\n",
        "MAPA_POSICIONES = {\n",
        "    'GK': 'GK',\n",
        "    'CB': 'DEF', 'LB': 'DEF', 'RB': 'DEF', 'LWB': 'DEF', 'RWB': 'DEF',\n",
        "    'CM': 'MID', 'CAM': 'MID', 'CDM': 'MID', 'LM': 'MID', 'RM': 'MID',\n",
        "    'ST': 'FWD', 'CF': 'FWD', 'LW': 'FWD', 'RW': 'FWD',\n",
        "    'LS': 'FWD', 'RS': 'FWD', 'LF': 'FWD', 'RF': 'FWD',\n",
        "}\n",
        "\n",
        "def categorizar_posicion(pos_str):\n",
        "    if pd.isna(pos_str):\n",
        "        return None\n",
        "    pos_principal = pos_str.split(',')[0].strip()\n",
        "    return MAPA_POSICIONES.get(pos_principal, 'MID')\n",
        "\n",
        "df['position_cat'] = df['player_positions'].apply(categorizar_posicion)\n",
        "df = df[df['position_cat'].notna()].copy()\n",
        "\n",
        "dist = df['position_cat'].value_counts()\n",
        "colores_pos = {'GK': 'gold', 'DEF': 'royalblue', 'MID': 'forestgreen', 'FWD': 'crimson'}\n",
        "\n",
        "fig, ax = plt.subplots(figsize=(8, 5))\n",
        "bars = ax.bar(dist.index, dist.values,\n",
        "              color=[colores_pos[p] for p in dist.index],\n",
        "              edgecolor='black', linewidth=0.8)\n",
        "for bar, val in zip(bars, dist.values):\n",
        "    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 20,\n",
        "            f'{val:,}\\n({val/len(df)*100:.1f}%)', ha='center', va='bottom', fontsize=11)\n",
        "ax.set_xlabel('Posición', fontsize=12)\n",
        "ax.set_ylabel('Número de jugadores', fontsize=12)\n",
        "ax.set_title('Distribución de Jugadores por Posición', fontsize=14, fontweight='bold')\n",
        "plt.tight_layout()\n",
        "plt.show()\n",
        "\n",
        "print(f'\\nDataset final: {len(df):,} jugadores con {len(dist)} categorías de posición')\n"
      ]
    },
    {
      "cell_type": "markdown",
      "id": "c10",
      "metadata": {
        "id": "c10"
      },
      "source": [
        "---\n",
        "## EJERCICIO 1: EDA del Target + Preparación Final\n",
        "**Dificultad:** ⭐⭐☆ (Media) — **15 puntos**\n",
        "\n",
        "---\n",
        "\n",
        "### Contexto\n",
        "\n",
        "El valor de mercado (`value_eur`) tiene una distribución muy sesgada a la derecha:\n",
        "unos pocos jugadores (Mbappé, Haaland...) tienen valores enormes mientras la gran\n",
        "mayoría vale mucho menos. Esto dificulta el aprendizaje de los modelos.\n",
        "\n",
        "**Solución:** transformación logarítmica → `y_log = np.log1p(value_eur)`\n",
        "\n",
        "Para convertir predicciones de vuelta a euros: `np.expm1(y_pred_log)`\n",
        "\n",
        "### Instrucciones\n",
        "\n",
        "1. **TODO 1.1** — Grafica side-by-side la distribución de `value_eur`:\n",
        "   - Izquierda: histograma de `value_eur` (escala normal)\n",
        "   - Derecha: histograma de `np.log1p(value_eur)`\n",
        "   - Añade títulos, etiquetas y línea de la media en cada gráfica\n",
        "\n",
        "2. **TODO 1.2** — Define `FEATURES_NUMERICAS` (la lista de columnas features)\n",
        "\n",
        "3. **TODO 1.3** — Crea los 4 splits (`X_train_r`, `X_test_r`, `X_train_c`, `X_test_c`)\n",
        "   - Para regresión: target = `np.log1p(df['value_eur'])`\n",
        "   - Para clasificación: target = `df['position_cat']`, usa `stratify=y_clf`\n",
        "   - Ambos con `test_size=0.2`\n"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "id": "c11",
      "metadata": {
        "id": "c11"
      },
      "outputs": [],
      "source": [
        "# ╔══════════════════════════════════════════════════════════╗\n",
        "# ║  EJERCICIO 1: EDA del Target + Preparación Final         ║\n",
        "# ╚══════════════════════════════════════════════════════════╝\n",
        "\n",
        "# ── TODO 1.1: Distribución de value_eur vs log1p(value_eur) ──────────────────\n",
        "fig, axes = plt.subplots(1, 2, figsize=(14, 5))\n",
        "\n",
        "# Tu código aquí:\n",
        "# axes[0].hist(...)   → value_eur sin transformar\n",
        "# axes[1].hist(...)   → np.log1p(df['value_eur'])\n",
        "# Añade título, etiquetas y línea de la media\n",
        "\n",
        "plt.tight_layout()\n",
        "plt.show()\n",
        "\n",
        "\n",
        "# ── TODO 1.2: Features numéricas ─────────────────────────────────────────────\n",
        "FEATURES_NUMERICAS = [\n",
        "    # Tu código aquí — lista las columnas que usarás como features\n",
        "    # Ejemplo: 'age', 'overall', 'potential', ...\n",
        "]\n",
        "print(f'Features seleccionadas: {FEATURES_NUMERICAS}')\n",
        "\n",
        "\n",
        "# ── TODO 1.3: Splits train/test ───────────────────────────────────────────────\n",
        "\n",
        "# — Regresión —\n",
        "X_reg = df[FEATURES_NUMERICAS].copy()\n",
        "y_reg = None  # TODO: np.log1p(df['value_eur'])\n",
        "\n",
        "X_train_r, X_test_r, y_train_r, y_test_r = None, None, None, None  # TODO: train_test_split(...)\n",
        "\n",
        "# — Clasificación —\n",
        "X_clf = df[FEATURES_NUMERICAS].copy()\n",
        "y_clf = None  # TODO: df['position_cat']\n",
        "\n",
        "X_train_c, X_test_c, y_train_c, y_test_c = None, None, None, None  # TODO: train_test_split(..., stratify=y_clf)\n",
        "\n",
        "\n",
        "# ── VERIFICACIÓN ─────────────────────────────────────────────────────────────\n",
        "print('=' * 55)\n",
        "print('VERIFICACIÓN EJERCICIO 1')\n",
        "print('=' * 55)\n",
        "print(f'  Features:          {len(FEATURES_NUMERICAS)} columnas')\n",
        "print(f'  Train regresión:   {X_train_r.shape[0]:,} muestras')\n",
        "print(f'  Test  regresión:   {X_test_r.shape[0]:,} muestras')\n",
        "print(f'  Train clasificación: {X_train_c.shape[0]:,} muestras')\n",
        "print(f'  Test  clasificación: {X_test_c.shape[0]:,} muestras')\n",
        "print(f'\\n  Distribución posiciones (test):')\n",
        "print(y_test_c.value_counts(normalize=True).round(3))\n"
      ]
    },
    {
      "cell_type": "markdown",
      "id": "c12",
      "metadata": {
        "id": "c12"
      },
      "source": [
        "---\n",
        "# Parte 3: Regresión — ¿Cuánto vale un jugador?\n",
        "\n",
        "\u003E **Conexión con Notebook 02** — Aplicamos modelos de regresión para predecir el valor de\n",
        "\u003E mercado. El target es `log1p(value_eur)` para manejar la distribución sesgada.\n",
        "\u003E Para interpretar las predicciones en euros reales: `np.expm1(y_pred)`.\n",
        "\n",
        "La celda siguiente contiene las **funciones auxiliares** y el modelo de **Regresión Lineal**\n",
        "como ejemplo completo. Estúdialos antes de implementar los ejercicios.\n"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "id": "c13",
      "metadata": {
        "id": "c13"
      },
      "outputs": [],
      "source": [
        "# === [CÓDIGO COMPLETO] FUNCIONES AUXILIARES + REGRESIÓN LINEAL ===\n",
        "\n",
        "resultados_reg = []   # Acumula resultados para comparación final\n",
        "\n",
        "def evaluar_regresion(y_real, y_pred, nombre):\n",
        "    rmse = np.sqrt(mean_squared_error(y_real, y_pred))\n",
        "    mae  = mean_absolute_error(y_real, y_pred)\n",
        "    r2   = r2_score(y_real, y_pred)\n",
        "    print(f\"\\n{'='*50}\\n {nombre}\\n{'='*50}\")\n",
        "    print(f'  RMSE: {rmse:.4f}')\n",
        "    print(f'  MAE:  {mae:.4f}')\n",
        "    print(f'  R2:   {r2:.4f}  ({r2*100:.1f}% varianza explicada)')\n",
        "    resultados_reg.append({'Modelo': nombre, 'RMSE': rmse, 'MAE': mae, 'R2': r2})\n",
        "    return r2\n",
        "\n",
        "\n",
        "# ── Regresión Lineal (modelo base) ───────────────────────────────────────────\n",
        "pipe_lr_r = Pipeline([\n",
        "    ('scaler', StandardScaler()),\n",
        "    ('modelo', LinearRegression())\n",
        "])\n",
        "pipe_lr_r.fit(X_train_r, y_train_r)\n",
        "y_pred_lr_r = pipe_lr_r.predict(X_test_r)\n",
        "\n",
        "r2_lr = evaluar_regresion(y_test_r, y_pred_lr_r, 'Regresión Lineal')\n",
        "\n",
        "# Gráficas: Pred vs Real + Coeficientes\n",
        "fig, axes = plt.subplots(1, 2, figsize=(14, 5))\n",
        "\n",
        "lim = [y_test_r.min() - 0.3, y_test_r.max() + 0.3]\n",
        "axes[0].scatter(y_test_r, y_pred_lr_r, alpha=0.3, s=15, color='cornflowerblue')\n",
        "axes[0].plot(lim, lim, 'r--', linewidth=2, label='Predicción perfecta')\n",
        "axes[0].set_xlabel('log(1+value_eur) Real', fontsize=11)\n",
        "axes[0].set_ylabel('log(1+value_eur) Predicho', fontsize=11)\n",
        "axes[0].set_title(f'Regresión Lineal — R² = {r2_lr:.3f}', fontsize=12)\n",
        "axes[0].legend()\n",
        "\n",
        "coefs = pipe_lr_r.named_steps['modelo'].coef_\n",
        "colores = ['forestgreen' if c \u003E 0 else 'crimson' for c in coefs]\n",
        "axes[1].barh(FEATURES_NUMERICAS, coefs, color=colores, edgecolor='gray', alpha=0.8)\n",
        "axes[1].axvline(0, color='black', linewidth=0.8)\n",
        "axes[1].set_xlabel('Coeficiente estandarizado', fontsize=11)\n",
        "axes[1].set_title('Importancia de variables (Regresión Lineal)', fontsize=12)\n",
        "\n",
        "plt.tight_layout()\n",
        "plt.show()\n",
        "print('Verde = aumenta el valor | Rojo = disminuye el valor')\n"
      ]
    },
    {
      "cell_type": "markdown",
      "id": "c14",
      "metadata": {
        "id": "c14"
      },
      "source": [
        "---\n",
        "## EJERCICIO 2: Árbol de Decisión para Regresión\n",
        "**Dificultad:** ⭐⭐☆ (Media) — **15 puntos**\n",
        "\n",
        "---\n",
        "\n",
        "Los árboles de decisión dividen el espacio de features en regiones rectangulares y predicen\n",
        "el **promedio** del target en cada región. Capturan relaciones **no lineales**, a diferencia\n",
        "de la regresión lineal.\n",
        "\n",
        "**Riesgo:** sin límites de profundidad el árbol memoriza el set de entrenamiento (overfitting).\n",
        "\n",
        "### Instrucciones\n",
        "\n",
        "1. **TODO 2.1** — Crea y entrena un `Pipeline([('scaler', StandardScaler()), ('modelo', DecisionTreeRegressor(...))])`\n",
        "   - Usa `max_depth=6` y `min_samples_leaf=10` como punto de partida\n",
        "2. **TODO 2.2** — Genera predicciones y llama a `evaluar_regresion()`\n",
        "3. **TODO 2.3** — Grafica Predicciones vs Valores Reales (igual al ejemplo de arriba)\n",
        "4. **TODO 2.4** *(Bonus)* — Grafica R² en test para `max_depth` de 1 a 15\n"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "id": "c15",
      "metadata": {
        "id": "c15"
      },
      "outputs": [],
      "source": [
        "# ╔══════════════════════════════════════════════════════════╗\n",
        "# ║  EJERCICIO 2: Árbol de Decisión para Regresión           ║\n",
        "# ╚══════════════════════════════════════════════════════════╝\n",
        "\n",
        "# ── TODO 2.1: Pipeline con DecisionTreeRegressor ─────────────────────────────\n",
        "pipe_dt_r = None  # Tu código aquí\n",
        "# pipe_dt_r.fit(...)\n",
        "\n",
        "\n",
        "# ── TODO 2.2: Predicciones + evaluación ──────────────────────────────────────\n",
        "y_pred_dt_r = None  # Tu código aquí\n",
        "\n",
        "r2_dt = evaluar_regresion(y_test_r, y_pred_dt_r, 'Árbol de Decisión (Regresión)')\n",
        "\n",
        "\n",
        "# ── TODO 2.3: Gráfica Predicciones vs Real ───────────────────────────────────\n",
        "fig, ax = plt.subplots(figsize=(7, 5))\n",
        "# Tu código aquí\n",
        "ax.set_xlabel('log(1+value_eur) Real', fontsize=11)\n",
        "ax.set_ylabel('log(1+value_eur) Predicho', fontsize=11)\n",
        "ax.set_title(f'Árbol de Decisión — R² = {r2_dt:.3f}', fontsize=12)\n",
        "plt.tight_layout()\n",
        "plt.show()\n",
        "\n",
        "\n",
        "# ── TODO 2.4 (Bonus): R² vs max_depth ────────────────────────────────────────\n",
        "profundidades = range(1, 16)\n",
        "r2_train_list = []\n",
        "r2_test_list  = []\n",
        "\n",
        "for depth in profundidades:\n",
        "    pipe_temp = Pipeline([('scaler', StandardScaler()),\n",
        "                          ('modelo', DecisionTreeRegressor(max_depth=depth, random_state=RANDOM_STATE))])\n",
        "    # Tu código aquí: fit, predice train y test, calcula r2 para cada uno\n",
        "    pass\n",
        "\n",
        "# Grafica r2_train_list y r2_test_list vs profundidades\n"
      ]
    },
    {
      "cell_type": "markdown",
      "id": "c16",
      "metadata": {
        "id": "c16"
      },
      "source": [
        "---\n",
        "## EJERCICIO 3: KNN para Regresión\n",
        "**Dificultad:** ⭐⭐☆ (Media) — **15 puntos**\n",
        "\n",
        "---\n",
        "\n",
        "KNN predice el valor de un jugador promediando los valores de sus K \"vecinos\" más similares.\n",
        "La estandarización es **obligatoria** (KNN calcula distancias; sin escalar, variables con\n",
        "rango grande dominarían).\n",
        "\n",
        "### Instrucciones\n",
        "\n",
        "1. **TODO 3.1** — `Pipeline` con `StandardScaler` + `KNeighborsRegressor(n_neighbors=5)`\n",
        "2. **TODO 3.2** — Predicciones y `evaluar_regresion()`\n",
        "3. **TODO 3.3** — Grafica R² en test para K de 1 a 30 e identifica el mejor K\n",
        "\n",
        "\u003E **Recuerda:** K=1 → sobreajuste perfecto en train | K grande → subajuste\n"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "id": "c17",
      "metadata": {
        "id": "c17"
      },
      "outputs": [],
      "source": [
        "# ╔══════════════════════════════════════════════════════════╗\n",
        "# ║  EJERCICIO 3: KNN para Regresión                         ║\n",
        "# ╚══════════════════════════════════════════════════════════╝\n",
        "\n",
        "# ── TODO 3.1: Pipeline con KNeighborsRegressor ───────────────────────────────\n",
        "pipe_knn_r = None  # Tu código aquí\n",
        "# pipe_knn_r.fit(...)\n",
        "\n",
        "\n",
        "# ── TODO 3.2: Predicciones + evaluación ──────────────────────────────────────\n",
        "y_pred_knn_r = None  # Tu código aquí\n",
        "\n",
        "r2_knn_r = evaluar_regresion(y_test_r, y_pred_knn_r, 'KNN Regresión (K=5)')\n",
        "\n",
        "\n",
        "# ── TODO 3.3: Gráfica R² vs K ────────────────────────────────────────────────\n",
        "k_valores  = range(1, 31)\n",
        "r2_por_k   = []\n",
        "\n",
        "for k in k_valores:\n",
        "    pipe_temp = Pipeline([('scaler', StandardScaler()),\n",
        "                          ('modelo', KNeighborsRegressor(n_neighbors=k))])\n",
        "    # Tu código aquí: entrena, predice en test, calcula R²\n",
        "    pass\n",
        "\n",
        "fig, ax = plt.subplots(figsize=(10, 5))\n",
        "# ax.plot(list(k_valores), r2_por_k, 's-', color='darkorange', linewidth=2)\n",
        "ax.set_xlabel('Número de vecinos K', fontsize=12)\n",
        "ax.set_ylabel('R² en test', fontsize=12)\n",
        "ax.set_title('KNN Regresión: R² según K', fontsize=13)\n",
        "plt.tight_layout()\n",
        "plt.show()\n",
        "\n",
        "mejor_k_r = None  # Tu código: k con mayor R²\n",
        "print(f'Mejor K para regresión: {mejor_k_r}')\n"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "id": "c18",
      "metadata": {
        "id": "c18"
      },
      "outputs": [],
      "source": [
        "# === [CÓDIGO COMPLETO] COMPARACIÓN DE MODELOS DE REGRESIÓN ===\n",
        "\n",
        "df_reg = pd.DataFrame(resultados_reg).round(4)\n",
        "\n",
        "print('=' * 60)\n",
        "print('COMPARACIÓN — MODELOS DE REGRESIÓN (FIFA 23)')\n",
        "print('Target: log(1 + value_eur)')\n",
        "print('=' * 60)\n",
        "print()\n",
        "print(df_reg.to_string(index=False))\n",
        "\n",
        "mejor = df_reg.loc[df_reg['R2'].idxmax()]\n",
        "print(f'\\nMejor modelo por R2: {mejor[\"Modelo\"]} (R2 = {mejor[\"R2\"]:.4f})')\n",
        "\n",
        "fig, axes = plt.subplots(1, 3, figsize=(15, 5))\n",
        "colores = ['cornflowerblue', 'forestgreen', 'darkorange'][:len(df_reg)]\n",
        "for ax, metrica in zip(axes, ['RMSE', 'MAE', 'R2']):\n",
        "    bars = ax.bar(df_reg['Modelo'], df_reg[metrica], color=colores, edgecolor='black')\n",
        "    for bar, val in zip(bars, df_reg[metrica]):\n",
        "        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.003,\n",
        "                f'{val:.3f}', ha='center', va='bottom', fontsize=10)\n",
        "    ax.set_title(metrica, fontsize=13, fontweight='bold')\n",
        "    ax.tick_params(axis='x', rotation=25)\n",
        "\n",
        "plt.suptitle('Comparación de Modelos de Regresión — FIFA 23', fontsize=14, fontweight='bold')\n",
        "plt.tight_layout()\n",
        "plt.show()\n"
      ]
    },
    {
      "cell_type": "markdown",
      "id": "c19",
      "metadata": {
        "id": "c19"
      },
      "source": [
        "---\n",
        "# Parte 4: Clasificación — ¿Cuál es la posición del jugador?\n",
        "\n",
        "\u003E **Conexión con Notebook 03** — Predecimos la posición (GK/DEF/MID/FWD) a partir de los stats.\n",
        "\u003E Es clasificación **multiclase** (4 categorías). Las métricas clave son Accuracy, F1 y\n",
        "\u003E la Matriz de Confusión.\n",
        "\n",
        "La celda siguiente implementa la **Regresión Logística** como ejemplo completo.\n"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "id": "c20",
      "metadata": {
        "id": "c20"
      },
      "outputs": [],
      "source": [
        "# === [CÓDIGO COMPLETO] REGRESIÓN LOGÍSTICA PARA CLASIFICACIÓN ===\n",
        "\n",
        "resultados_clf = []\n",
        "\n",
        "def evaluar_clasificacion(y_real, y_pred, nombre):\n",
        "    acc = accuracy_score(y_real, y_pred)\n",
        "    print(f\"\\n{'='*55}\\n {nombre}\\n{'='*55}\")\n",
        "    print(f'  Accuracy: {acc:.4f}  ({acc*100:.1f}%)')\n",
        "    print('\\n  Reporte por clase:')\n",
        "    print(classification_report(y_real, y_pred))\n",
        "    resultados_clf.append({'Modelo': nombre, 'Accuracy': acc})\n",
        "    return acc\n",
        "\n",
        "\n",
        "pipe_lr_c = Pipeline([\n",
        "    ('scaler', StandardScaler()),\n",
        "    ('modelo', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))\n",
        "])\n",
        "pipe_lr_c.fit(X_train_c, y_train_c)\n",
        "y_pred_lr_c = pipe_lr_c.predict(X_test_c)\n",
        "\n",
        "acc_lr_c = evaluar_clasificacion(y_test_c, y_pred_lr_c, 'Regresión Logística')\n",
        "\n",
        "# Matriz de confusión\n",
        "clases = ['GK', 'DEF', 'MID', 'FWD']\n",
        "fig, ax = plt.subplots(figsize=(7, 6))\n",
        "cm = confusion_matrix(y_test_c, y_pred_lr_c, labels=clases)\n",
        "sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,\n",
        "            xticklabels=clases, yticklabels=clases, annot_kws={'size': 13})\n",
        "ax.set_xlabel('Predicción', fontsize=12)\n",
        "ax.set_ylabel('Real', fontsize=12)\n",
        "ax.set_title(f'Matriz de Confusión — Reg. Logística\\nAccuracy: {acc_lr_c:.3f}', fontsize=13)\n",
        "plt.tight_layout()\n",
        "plt.show()\n"
      ]
    },
    {
      "cell_type": "markdown",
      "id": "c21",
      "metadata": {
        "id": "c21"
      },
      "source": [
        "---\n",
        "## EJERCICIO 4: Árbol de Decisión + KNN para Clasificación\n",
        "**Dificultad:** ⭐⭐⭐ (Alta) — **20 puntos**\n",
        "\n",
        "---\n",
        "\n",
        "### Parte A — Árbol de Decisión Clasificador\n",
        "\n",
        "1. `Pipeline` con `StandardScaler` + `DecisionTreeClassifier`\n",
        "2. Limita `max_depth` (entre 8 y 15) para evitar sobreajuste\n",
        "3. Llama a `evaluar_clasificacion()` y grafica la matriz de confusión\n",
        "\n",
        "### Parte B — KNN Clasificador\n",
        "\n",
        "1. `Pipeline` con `StandardScaler` + `KNeighborsClassifier(n_neighbors=9)`\n",
        "2. Llama a `evaluar_clasificacion()` y grafica la matriz de confusión\n",
        "\n",
        "### Análisis (escribe tus conclusiones en la celda Markdown siguiente):\n",
        "- ¿Cuál modelo supera a la Regresión Logística?\n",
        "- ¿Qué posición es más difícil de clasificar? ¿Por qué?\n",
        "- ¿Los porteros (GK) son fáciles de identificar?\n",
        "\n",
        "\u003E **Observación esperada:** Los GK se distinguen fácilmente. La confusión suele ser\n",
        "\u003E entre MID y FWD, que tienen stats similares.\n"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "id": "c22",
      "metadata": {
        "id": "c22"
      },
      "outputs": [],
      "source": [
        "# ╔══════════════════════════════════════════════════════════╗\n",
        "# ║  EJERCICIO 4: Árbol de Decisión + KNN Clasificación      ║\n",
        "# ╚══════════════════════════════════════════════════════════╝\n",
        "\n",
        "clases = ['GK', 'DEF', 'MID', 'FWD']\n",
        "\n",
        "# ════════════════════════════════════════════════\n",
        "# PARTE A: ÁRBOL DE DECISIÓN\n",
        "# ════════════════════════════════════════════════\n",
        "\n",
        "# ── TODO 4A.1: Pipeline DecisionTreeClassifier ───────────────────────────────\n",
        "pipe_dt_c = None  # Tu código aquí\n",
        "# pipe_dt_c.fit(...)\n",
        "\n",
        "y_pred_dt_c = None  # Tu código aquí\n",
        "acc_dt_c = evaluar_clasificacion(y_test_c, y_pred_dt_c, 'Árbol de Decisión (Clasificación)')\n",
        "\n",
        "# ── TODO 4A.2: Matriz de confusión ───────────────────────────────────────────\n",
        "fig, ax = plt.subplots(figsize=(7, 6))\n",
        "# cm = confusion_matrix(y_test_c, y_pred_dt_c, labels=clases)\n",
        "# sns.heatmap(...)\n",
        "plt.tight_layout()\n",
        "plt.show()\n",
        "\n",
        "\n",
        "# ════════════════════════════════════════════════\n",
        "# PARTE B: KNN CLASIFICADOR\n",
        "# ════════════════════════════════════════════════\n",
        "\n",
        "# ── TODO 4B.1: Pipeline KNeighborsClassifier ─────────────────────────────────\n",
        "pipe_knn_c = None  # Tu código aquí\n",
        "# pipe_knn_c.fit(...)\n",
        "\n",
        "y_pred_knn_c = None  # Tu código aquí\n",
        "acc_knn_c = evaluar_clasificacion(y_test_c, y_pred_knn_c, 'KNN Clasificación (K=9)')\n",
        "\n",
        "# ── TODO 4B.2: Matriz de confusión ───────────────────────────────────────────\n",
        "fig, ax = plt.subplots(figsize=(7, 6))\n",
        "# Tu código aquí\n",
        "plt.tight_layout()\n",
        "plt.show()\n"
      ]
    },
    {
      "cell_type": "markdown",
      "id": "c23",
      "metadata": {
        "id": "c23"
      },
      "source": [
        "### Análisis del Ejercicio 4 — Escribe tus conclusiones aquí\n",
        "\n",
        "**Edita esta celda y responde:**\n",
        "\n",
        "1. ¿Cuál modelo obtuvo mayor accuracy? ¿Cuánto supera a la Regresión Logística?\n",
        "\n",
        "2. ¿Qué posición fue la más difícil de clasificar? Apoya tu respuesta con los números de la matriz.\n",
        "\n",
        "3. ¿Los porteros (GK) fueron fáciles de identificar? ¿Qué stats crees que los distinguen?\n",
        "\n",
        "4. ¿Hubo confusión entre algún par de posiciones? ¿Tiene sentido futbolísticamente?\n",
        "\n",
        "*Tu respuesta aquí...*\n"
      ]
    },
    {
      "cell_type": "markdown",
      "id": "c26",
      "metadata": {
        "id": "c26"
      },
      "source": [
        "---\n",
        "## EJERCICIO 5: Random Forest para Clasificación + Importancia de Variables\n",
        "**Dificultad:** ⭐⭐⭐ (Alta) — **20 puntos**\n",
        "\n",
        "---\n",
        "\n",
        "### Instrucciones\n",
        "\n",
        "1. **TODO 5.1** — `Pipeline` con `RandomForestClassifier(n_estimators=200, random_state=42)`\n",
        "2. **TODO 5.2** — Predice, llama a `evaluar_clasificacion()` y grafica la matriz de confusión\n",
        "3. **TODO 5.3** — Extrae `feature_importances_` y grafica las 14 variables ordenadas\n",
        "4. **TODO 5.4** — Responde: ¿coinciden las variables más importantes con las de la regresión?\n",
        "\n",
        "\u003E **Acceso a importancias:** `pipe_rf_c.named_steps['modelo'].feature_importances_`\n",
        "\n",
        "\u003E **Esperas encontrar:** `defending` importante para GK/DEF; `shooting` y `pace` para FWD.\n"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "id": "c27",
      "metadata": {
        "id": "c27"
      },
      "outputs": [],
      "source": [
        "# ╔══════════════════════════════════════════════════════════╗\n",
        "# ║  EJERCICIO 5: Random Forest Clasificación               ║\n",
        "# ╚══════════════════════════════════════════════════════════╝\n",
        "\n",
        "# ── TODO 5.1: Pipeline RandomForestClassifier ────────────────────────────────\n",
        "pipe_rf_c = None  # Tu código aquí\n",
        "# pipe_rf_c.fit(...)\n",
        "\n",
        "\n",
        "# ── TODO 5.2: Predicciones, evaluación y matriz de confusión ─────────────────\n",
        "y_pred_rf_c = None  # Tu código aquí\n",
        "acc_rf_c = evaluar_clasificacion(y_test_c, y_pred_rf_c, 'Random Forest (Clasificación)')\n",
        "\n",
        "fig, ax = plt.subplots(figsize=(7, 6))\n",
        "# Tu código aquí: matriz de confusión\n",
        "plt.tight_layout()\n",
        "plt.show()\n",
        "\n",
        "\n",
        "# ── TODO 5.3: Importancia de variables ───────────────────────────────────────\n",
        "importancias_c = None  # pipe_rf_c.named_steps['modelo'].feature_importances_\n",
        "\n",
        "df_imp_c = pd.DataFrame({\n",
        "    'Variable': FEATURES_NUMERICAS,\n",
        "    'Importancia': importancias_c\n",
        "}).sort_values('Importancia', ascending=True)\n",
        "\n",
        "fig, ax = plt.subplots(figsize=(8, 6))\n",
        "# Tu código aquí: barh con las importancias\n",
        "ax.set_xlabel('Importancia relativa', fontsize=12)\n",
        "ax.set_title('Importancia de Variables — RF Clasificación (Posición)', fontsize=13)\n",
        "plt.tight_layout()\n",
        "plt.show()\n",
        "\n",
        "print('Top 5 variables para predecir la posición:')\n",
        "# Tu código aquí\n"
      ]
    },
    {
      "cell_type": "markdown",
      "id": "c28",
      "metadata": {
        "id": "c28"
      },
      "source": [
        "---\n",
        "# Parte 6: Validación y Ajuste del Mejor Modelo\n",
        "\n",
        "\u003E **Conexión con Notebook 05** — Usamos validación cruzada para evaluar el modelo de forma\n",
        "\u003E más robusta, y luego `GridSearchCV` para encontrar los hiperparámetros óptimos.\n",
        "\n",
        "La celda siguiente muestra la validación cruzada del Random Forest (regresión) como ejemplo.\n"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "id": "c29",
      "metadata": {
        "id": "c29"
      },
      "outputs": [],
      "source": [
        "# === [CÓDIGO COMPLETO] VALIDACIÓN CRUZADA ===\n",
        "\n",
        "print('=' * 60)\n",
        "print('VALIDACIÓN CRUZADA 5-FOLD — Random Forest (Regresión)')\n",
        "print('=' * 60)\n",
        "print()\n",
        "print('Un único train/test split depende de la suerte del sorteo.')\n",
        "print('La validación cruzada evalúa el modelo en 5 particiones distintas.')\n",
        "\n",
        "cv_scores = cross_val_score(pipe_rf_r, X_reg, y_reg, cv=5, scoring='r2', n_jobs=-1)\n",
        "\n",
        "print(f'\\nScores por fold: {cv_scores.round(4)}')\n",
        "print(f'Media:           {cv_scores.mean():.4f}')\n",
        "print(f'Desv. estándar:  {cv_scores.std():.4f}')\n",
        "print(f'Resultado final: R² = {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')\n",
        "\n",
        "fig, ax = plt.subplots(figsize=(8, 4))\n",
        "ax.bar(range(1, 6), cv_scores, color='forestgreen', edgecolor='black', alpha=0.8)\n",
        "ax.axhline(cv_scores.mean(), color='red', linestyle='--', linewidth=2,\n",
        "           label=f'Media: {cv_scores.mean():.4f}')\n",
        "ax.fill_between(range(0, 7),\n",
        "                cv_scores.mean() - cv_scores.std(),\n",
        "                cv_scores.mean() + cv_scores.std(),\n",
        "                alpha=0.15, color='red', label='±1 desv. std.')\n",
        "ax.set_xlabel('Fold', fontsize=12)\n",
        "ax.set_ylabel('R²', fontsize=12)\n",
        "ax.set_title('Validación Cruzada 5-Fold — Random Forest Regresión', fontsize=13)\n",
        "ax.set_xlim(0.5, 5.5)\n",
        "ax.legend(fontsize=11)\n",
        "plt.tight_layout()\n",
        "plt.show()\n"
      ]
    },
    {
      "cell_type": "markdown",
      "id": "c30",
      "metadata": {
        "id": "c30"
      },
      "source": [
        "---\n",
        "## EJERCICIO 6: GridSearchCV — Ajuste del Mejor Modelo de Clasificación\n",
        "**Dificultad:** ⭐⭐⭐ (Alta) — **15 puntos**\n",
        "\n",
        "---\n",
        "\n",
        "### Instrucciones\n",
        "\n",
        "1. **TODO 6.1** — Define la grilla con al menos 3 hiperparámetros\n",
        "\n",
        "```python\n",
        "param_grid = {\n",
        "    'modelo__n_estimators': [50, 100, 200],\n",
        "    'modelo__max_depth':    [10, 20, None],\n",
        "    'modelo__min_samples_leaf': [1, 2, 5]\n",
        "}\n",
        "```\n",
        "\n",
        "\u003E ⚠️ En un `Pipeline`, los parámetros del estimador se prefijan con el nombre del paso:\n",
        "\u003E `'modelo__n_estimators'` (no solo `'n_estimators'`).\n",
        "\n",
        "2. **TODO 6.2** — Ejecuta `GridSearchCV` con `cv=StratifiedKFold(5)` y `scoring='accuracy'`\n",
        "\n",
        "3. **TODO 6.3** — Compara el accuracy antes y después del ajuste (validación cruzada)\n",
        "\n",
        "4. **TODO 6.4** *(Bonus)* — Prueba `RandomizedSearchCV` con `n_iter=20` y compara tiempos\n",
        "\n",
        "\u003E ⏱️ **Tiempo estimado:** 3–10 min. Empieza con la grilla pequeña sugerida.\n"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "id": "c31",
      "metadata": {
        "id": "c31"
      },
      "outputs": [],
      "source": [
        "# ╔══════════════════════════════════════════════════════════╗\n",
        "# ║  EJERCICIO 6: GridSearchCV                               ║\n",
        "# ╚══════════════════════════════════════════════════════════╝\n",
        "\n",
        "# ── TODO 6.1: Grilla de hiperparámetros ──────────────────────────────────────\n",
        "param_grid = {\n",
        "    # Tu código aquí (al menos 3 parámetros)\n",
        "    # 'modelo__n_estimators': [...],\n",
        "    # 'modelo__max_depth': [...],\n",
        "    # 'modelo__min_samples_leaf': [...]\n",
        "}\n",
        "\n",
        "n_combos = 1\n",
        "for vals in param_grid.values():\n",
        "    n_combos *= len(vals)\n",
        "print(f'Combinaciones: {n_combos} × 5 folds = {n_combos * 5} entrenamientos')\n",
        "\n",
        "\n",
        "# ── TODO 6.2: GridSearchCV ────────────────────────────────────────────────────\n",
        "pipe_base_grid = Pipeline([\n",
        "    ('scaler', StandardScaler()),\n",
        "    ('modelo', RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1))\n",
        "])\n",
        "\n",
        "grid_search = None  # Tu código aquí: GridSearchCV(...)\n",
        "\n",
        "print('Buscando mejores hiperparámetros...')\n",
        "# grid_search.fit(X_train_c, y_train_c)\n",
        "\n",
        "\n",
        "# ── TODO 6.3: Resultados y comparación ───────────────────────────────────────\n",
        "print('\\n' + '=' * 60)\n",
        "print('RESULTADOS GRID SEARCH')\n",
        "print('=' * 60)\n",
        "\n",
        "# Tu código: imprime grid_search.best_params_ y grid_search.best_score_\n",
        "\n",
        "acc_antes = cross_val_score(pipe_rf_c, X_clf, y_clf, cv=5,\n",
        "                             scoring='accuracy', n_jobs=-1).mean()\n",
        "acc_despues = None  # Tu código: CV del mejor estimador\n",
        "\n",
        "print(f'\\nAccuracy CV antes del ajuste:   {acc_antes:.4f}')\n",
        "print(f'Accuracy CV después del ajuste: {acc_despues:.4f}')\n",
        "print(f'Mejora: {(acc_despues - acc_antes)*100:+.2f} pp')\n"
      ]
    },
    {
      "cell_type": "code",
      "execution_count": null,
      "id": "c32",
      "metadata": {
        "id": "c32"
      },
      "outputs": [],
      "source": [
        "# === TABLA RESUMEN FINAL ===\n",
        "\n",
        "print('=' * 65)\n",
        "print('RESUMEN FINAL DEL PROYECTO — FIFA 23 Players')\n",
        "print('=' * 65)\n",
        "\n",
        "if resultados_reg:\n",
        "    print('\\n--- REGRESIÓN (Target: log(1+value_eur)) ---')\n",
        "    df_reg_f = pd.DataFrame(resultados_reg).round(4)\n",
        "    print(df_reg_f.to_string(index=False))\n",
        "    mejor_r = df_reg_f.loc[df_reg_f['R2'].idxmax()]\n",
        "    print(f'  Mejor: {mejor_r[\"Modelo\"]} — R² = {mejor_r[\"R2\"]:.4f}')\n",
        "\n",
        "if resultados_clf:\n",
        "    print('\\n--- CLASIFICACIÓN (Target: position_cat GK/DEF/MID/FWD) ---')\n",
        "    df_clf_f = pd.DataFrame(resultados_clf).round(4)\n",
        "    print(df_clf_f.to_string(index=False))\n",
        "    mejor_c = df_clf_f.loc[df_clf_f['Accuracy'].idxmax()]\n",
        "    print(f'  Mejor: {mejor_c[\"Modelo\"]} — Accuracy = {mejor_c[\"Accuracy\"]:.4f}')\n"
      ]
    },
    {
      "cell_type": "markdown",
      "id": "c33",
      "metadata": {
        "id": "c33"
      },
      "source": [
        "---\n",
        "# Criterios de Evaluación y Entrega\n",
        "\n",
        "## Rúbrica (100 puntos)\n",
        "\n",
        "| Ejercicio | Descripción | Puntos |\n",
        "|-----------|-------------|--------|\n",
        "| **Ejercicio 1** | EDA del target + splits correctos con stratify | 15 pts |\n",
        "| **Ejercicio 2** | Árbol de Decisión regresión + gráfica Pred vs Real | 15 pts |\n",
        "| **Ejercicio 3** | KNN regresión + gráfica R² vs K | 15 pts |\n",
        "| **Ejercicio 4** | Árbol + KNN clasificación + análisis escrito | 20 pts |\n",
        "| **Ejercicio 5** | Random Forest clasificación + gráfica importancia | 20 pts |\n",
        "| **Ejercicio 6** | GridSearchCV + comparación antes/después | 15 pts |\n",
        "| **Total** | | **100 pts** |\n",
        "\n",
        "### Criterios transversales (aplican a todos los ejercicios):\n",
        "- El código corre **sin errores** *(requisito mínimo para recibir puntos)*\n",
        "- Las gráficas tienen **título, etiquetas de ejes** y son legibles\n",
        "- Las métricas están **correctamente calculadas** y llamadas con `evaluar_*()`\n",
        "- Las **conclusiones escritas** demuestran comprensión del resultado\n",
        "\n",
        "## Bonus (hasta +10 puntos)\n",
        "\n",
        "| Actividad | Puntos |\n",
        "|-----------|--------|\n",
        "| Implementar `GradientBoostingClassifier` y comparar con RF | +3 pts |\n",
        "| Identificar jugadores mal clasificados y analizar por qué | +3 pts |\n",
        "| Encontrar el mejor K del Ejercicio 3 con validación cruzada (no solo test) | +2 pts |\n",
        "| Convertir las predicciones de regresión a euros reales con `np.expm1()` | +2 pts |\n",
        "\n",
        "## Instrucciones de entrega\n",
        "\n",
        "1. Ejecuta **todas las celdas** (Kernel → Restart & Run All) y verifica que no haya errores\n",
        "2. Guarda el notebook como: `Proyecto_FIFA_[TuNombre]_[TuCódigo].ipynb`\n",
        "3. Sube el archivo al enlace de entrega del curso\n",
        "\n",
        "---\n",
        "*Proyecto Integrador — Inteligencia Artificial y Aprendizaje Automático*\n"
      ]
    }
  ],
  "metadata": {
    "kernelspec": {
      "display_name": "Python 3",
      "language": "python",
      "name": "python3"
    },
    "language_info": {
      "codemirror_mode": {
        "name": "ipython",
        "version": 3
      },
      "file_extension": ".py",
      "mimetype": "text/x-python",
      "name": "python",
      "pygments_lexer": "ipython3",
      "version": "3.9.0"
    },
    "colab": {
      "provenance": []
    }
  },
  "nbformat": 4,
  "nbformat_minor": 5
}